In [1]:
# part1_load_prepare.py
import geopandas as gpd
import pandas as pd
import os

In [2]:
# ---------- UPDATE PATHS ----------
train_shp = r"D:\THESIS\Machine_Learning\training_70.shp"
test_shp  = r"D:\THESIS\Machine_Learning\testing_30.shp"
out_dir = r"D:\THESIS\Machine_Learning\ml_inputs"
os.makedirs(out_dir, exist_ok=True)

In [3]:
features = [
    "elevation","slope","lulc","ndvi","twi",
    "df_roads","df_rivers","rainfall","aspect","drainage_d"
]
label = "Class"  # make sure field name matches

In [4]:
# Load
train_gdf = gpd.read_file(train_shp)
test_gdf  = gpd.read_file(test_shp)

# Basic sanity checks
print("Train rows:", len(train_gdf), "Test rows:", len(test_gdf))
for f in features + [label]:
    if f not in train_gdf.columns:
        raise ValueError(f"Missing field in training: {f}")
    if f not in test_gdf.columns:
        raise ValueError(f"Missing field in testing: {f}")

Train rows: 1073 Test rows: 476


In [5]:
# Drop any rows with NaN or -9999 in the feature set
def clean_df(gdf):
    df = gdf[features + [label]].copy()
    # replace common nodata marker -9999 with NaN, then drop
    df = df.replace(-9999, pd.NA)
    df = df.dropna()
    return df

train_df = clean_df(train_gdf)
test_df  = clean_df(test_gdf)

print("After cleaning, Train rows:", len(train_df), "Test rows:", len(test_df))

After cleaning, Train rows: 1073 Test rows: 476


In [6]:
# Save cleaned CSVs for easy use with sklearn
train_csv = os.path.join(out_dir, "train_clean.csv")
test_csv  = os.path.join(out_dir, "test_clean.csv")
train_df.to_csv(train_csv, index=False)
test_df.to_csv(test_csv, index=False)

print("Wrote:", train_csv, test_csv)

Wrote: D:\THESIS\Machine_Learning\ml_inputs\train_clean.csv D:\THESIS\Machine_Learning\ml_inputs\test_clean.csv


In [7]:
# part2_train_models.py
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from joblib import dump
import os

In [8]:
# ---------- UPDATE PATHS ----------
in_dir = r"D:\THESIS\Machine_Learning\ml_inputs"
train_csv = os.path.join(in_dir, "train_clean.csv")
out_dir = r"D:\THESIS\Machine_Learning\ml_models"
os.makedirs(out_dir, exist_ok=True)

In [9]:
features = [
    "elevation","slope","lulc","ndvi","twi",
    "df_roads","df_rivers","rainfall","aspect","drainage_d"
]
categorical = ["lulc"]
numeric = [f for f in features if f not in categorical]

In [10]:
# Load data
train = pd.read_csv(train_csv)
X_train = train[features]
y_train = train["Class"].astype(int)

In [11]:
# Preprocessor
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse=False), categorical)
], remainder="drop")

In [12]:
# Pipelines
rf_pipe = Pipeline([("pre", preprocessor),
                    ("clf", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))])

gb_pipe = Pipeline([("pre", preprocessor),
                    ("clf", GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=42))])

svm_pipe = Pipeline([("pre", preprocessor),
                     ("clf", SVC(kernel="rbf", probability=True, C=1.0, gamma="scale", random_state=42))])

NameError: name 'preprocessor' is not defined

In [12]:
# Fit models
print("Fitting Random Forest...")
rf_pipe.fit(X_train, y_train)
dump(rf_pipe, os.path.join(out_dir, "rf_pipe.joblib"))

print("Fitting Gradient Boosting...")
gb_pipe.fit(X_train, y_train)
dump(gb_pipe, os.path.join(out_dir, "gb_pipe.joblib"))

print("Fitting SVM...")
svm_pipe.fit(X_train, y_train)
dump(svm_pipe, os.path.join(out_dir, "svm_pipe.joblib"))

print("Saved models to", out_dir)

Fitting Random Forest...


NameError: name 'rf_pipe' is not defined

In [15]:
# part3_evaluate.py
import pandas as pd
from joblib import load, dump
from sklearn.metrics import accuracy_score, precision_score, roc_auc_score, confusion_matrix, classification_report
import os
import numpy as np

In [16]:
# ---------- UPDATE PATHS ----------
in_dir = r"D:\THESIS\Machine_Learning\ml_inputs"
test_csv = os.path.join(in_dir, "test_clean.csv")
model_dir = r"D:\THESIS\Machine_Learning\ml_models"

In [17]:
test = pd.read_csv(test_csv)
features = [
    "elevation","slope","lulc","ndvi","twi",
    "df_roads","df_rivers","rainfall","aspect","drainage_d"
]
X_test = test[features]
y_test = test["Class"].astype(int)

In [18]:
models = {
    "RF": load(os.path.join(model_dir, "rf_pipe.joblib")),
    "GB": load(os.path.join(model_dir, "gb_pipe.joblib")),
    "SVM": load(os.path.join(model_dir, "svm_pipe.joblib"))
}

In [18]:
results = {}
for name, model in models.items():
    y_pred = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        # fallback
        y_proba = np.zeros(len(y_pred))
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_proba) if y_proba.sum() != 0 else None
    cm = confusion_matrix(y_test, y_pred)
    print(f"\n=== {name} ===")
    print("Accuracy:", acc)
    print("Precision:", prec)
    print("ROC AUC:", auc)
    print("Confusion matrix:\n", cm)
    print(classification_report(y_test, y_pred))
    results[name] = {"acc": acc, "prec": prec, "auc": auc}


=== RF ===
Accuracy: 0.8269230769230769
Precision: 0.7813620071684588
ROC AUC: 0.883339422725612
Confusion matrix:
 [[169  61]
 [ 20 218]]
              precision    recall  f1-score   support

           0       0.89      0.73      0.81       230
           1       0.78      0.92      0.84       238

    accuracy                           0.83       468
   macro avg       0.84      0.83      0.83       468
weighted avg       0.84      0.83      0.83       468


=== GB ===
Accuracy: 0.8247863247863247
Precision: 0.7846715328467153
ROC AUC: 0.8862623310193642
Confusion matrix:
 [[171  59]
 [ 23 215]]
              precision    recall  f1-score   support

           0       0.88      0.74      0.81       230
           1       0.78      0.90      0.84       238

    accuracy                           0.82       468
   macro avg       0.83      0.82      0.82       468
weighted avg       0.83      0.82      0.82       468


=== SVM ===
Accuracy: 0.8247863247863247
Precision: 0.7617449664

In [20]:
# Choose best by ROC AUC if available, else by precision
best = None
best_score = -1
for k,v in results.items():
    score = v["auc"] if v["auc"] is not None else v["prec"]
    if score is None:
        continue
    if score > best_score:
        best_score = score
        best = k

print("\nBest model by chosen metric is:", best, "score:", best_score)
best_model = models[best]
dump(best_model, os.path.join(model_dir, "best_flood_model.joblib"))
print("Saved best model as best_flood_model.joblib")

NameError: name 'results' is not defined

In [20]:
# part4_apply_model.py
import rasterio
from rasterio.windows import Window
import numpy as np
import pandas as pd
from joblib import load
import os
from tqdm import tqdm

In [21]:
# ---------- UPDATE PATHS ----------
model_path = r"D:\THESIS\Machine_Learning\ml_models\best_flood_model.joblib"
# map feature name to raster path, keys must match features order
raster_files = {
    "df_rivers": r"D:\THESIS\Machine_Learning\Predictor_Variables\df_rivers.tif",
    "df_roads": r"D:\THESIS\Machine_Learning\Predictor_Variables\df_roads.tif",
    "drainage_d": r"D:\THESIS\Machine_Learning\Predictor_Variables\drainage_density.tif",
    "elevation": r"D:\THESIS\Machine_Learning\Predictor_Variables\elevation.tif",
    "lulc": r"D:\THESIS\Machine_Learning\Predictor_Variables\lulc.tif",
    "ndvi": r"D:\THESIS\Machine_Learning\Predictor_Variables\ndvi.tif",
    "rainfall": r"D:\THESIS\Machine_Learning\Predictor_Variables\rainfall.tif",
    "slope": r"D:\THESIS\Machine_Learning\Predictor_Variables\slope.tif",
    "soil_type": r"D:\THESIS\Machine_Learning\Predictor_Variables\soil_type.tif",
    "twi": r"D:\THESIS\Machine_Learning\Predictor_Variables\twi.tif",
}
out_path = r"D:\THESIS\Machine_Learning\outputs\flood_probability.tif"
os.makedirs(os.path.dirname(out_path), exist_ok=True)

In [22]:
features = list(raster_files.keys())
model = load(model_path)

In [23]:
# Open rasters and ensure they align
srcs = [rasterio.open(raster_files[f]) for f in features]
# basic check: all same width, height, transform, crs
w = srcs[0].width; h = srcs[0].height; tr = srcs[0].transform; crs = srcs[0].crs
for s in srcs[1:]:
    if s.width != w or s.height != h:
        raise ValueError("Raster dimensions do not match")
    if s.transform != tr:
        raise ValueError("Raster transforms do not match")
    if s.crs != crs:
        raise ValueError("Raster CRS do not match")

ValueError: Raster dimensions do not match

In [24]:
# diagnostic_rasters.py
import rasterio
raster_files = {
    "df_rivers": r"D:\THESIS\Machine_Learning\Predictor_Variables\df_rivers.tif",
    "df_roads": r"D:\THESIS\Machine_Learning\Predictor_Variables\df_roads.tif",
    "drainage_d": r"D:\THESIS\Machine_Learning\Predictor_Variables\drainage_density.tif",
    "elevation": r"D:\THESIS\Machine_Learning\Predictor_Variables\elevation.tif",
    "lulc": r"D:\THESIS\Machine_Learning\Predictor_Variables\lulc.tif",
    "ndvi": r"D:\THESIS\Machine_Learning\Predictor_Variables\ndvi.tif",
    "rainfall": r"D:\THESIS\Machine_Learning\Predictor_Variables\rainfall.tif",
    "slope": r"D:\THESIS\Machine_Learning\Predictor_Variables\slope.tif",
    "soil_type": r"D:\THESIS\Machine_Learning\Predictor_Variables\soil_type.tif",
    "twi": r"D:\THESIS\Machine_Learning\Predictor_Variables\twi.tif",
}

for name, path in raster_files.items():
    with rasterio.open(path) as src:
        print(name)
        print("  path:", path)
        print("  crs:", src.crs)
        print("  width, height:", src.width, src.height)
        print("  transform:", src.transform)
        print("  bounds:", src.bounds)
        print("  nodata:", src.nodata)
        print()

df_rivers
  path: D:\THESIS\Machine_Learning\Predictor_Variables\df_rivers.tif
  crs: EPSG:32646
  width, height: 250 392
  transform: | 320.79, 0.00, 339028.61|
| 0.00,-320.79, 2543981.42|
| 0.00, 0.00, 1.00|
  bounds: BoundingBox(left=339028.60529999994, bottom=2418229.8124, right=419227.33310000016, top=2543981.4175904007)
  nodata: -3.4028230607370965e+38

df_roads
  path: D:\THESIS\Machine_Learning\Predictor_Variables\df_roads.tif
  crs: EPSG:32646
  width, height: 252 395
  transform: | 320.00, 0.00, 338776.94|
| 0.00,-320.00, 2544377.78|
| 0.00, 0.00, 1.00|
  bounds: BoundingBox(left=338776.94039999973, bottom=2417977.7837000005, right=419416.94039999973, top=2544377.7837000005)
  nodata: -3.4028230607370965e+38

drainage_d
  path: D:\THESIS\Machine_Learning\Predictor_Variables\drainage_density.tif
  crs: EPSG:32646
  width, height: 251 393
  transform: | 320.79, 0.00, 338868.21|
| 0.00,-320.79, 2544141.82|
| 0.00, 0.00, 1.00|
  bounds: BoundingBox(left=338868.20784439996, botto

In [25]:
# align_rasters_to_reference.py
import os
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject, calculate_default_transform

In [26]:
# ---------- UPDATE ----------
ref_raster = r"D:\THESIS\Machine_Learning\Predictor_Variables\elevation.tif"   # choose your reference raster, the one you want all others to match
raster_files = {
    "elevation": ref_raster,
    "df_rivers": r"D:\THESIS\Machine_Learning\Predictor_Variables\df_rivers.tif",
    "df_roads": r"D:\THESIS\Machine_Learning\Predictor_Variables\df_roads.tif",
    "drainage_d": r"D:\THESIS\Machine_Learning\Predictor_Variables\drainage_density.tif",
    "lulc": r"D:\THESIS\Machine_Learning\Predictor_Variables\lulc.tif",
    "ndvi": r"D:\THESIS\Machine_Learning\Predictor_Variables\ndvi.tif",
    "rainfall": r"D:\THESIS\Machine_Learning\Predictor_Variables\rainfall.tif",
    "slope": r"D:\THESIS\Machine_Learning\Predictor_Variables\slope.tif",
    "soil_type": r"D:\THESIS\Machine_Learning\Predictor_Variables\soil_type.tif",
    "twi": r"D:\THESIS\Machine_Learning\Predictor_Variables\twi.tif",
}
out_dir = r"D:\THESIS\Machine_Learning\rasters_aligned"
os.makedirs(out_dir, exist_ok=True)

In [27]:
# open reference to get desired grid
with rasterio.open(ref_raster) as ref:
    ref_meta = ref.meta.copy()
    ref_crs = ref.crs
    ref_transform = ref.transform
    ref_width = ref.width
    ref_height = ref.height
    ref_dtype = ref.dtypes[0]
    ref_nodata = ref.nodata

In [28]:
for name, path in raster_files.items():
    # Define output path
    out_path = os.path.join(out_dir, f"{name}_aligned.tif")

    # If reference raster, simply copy it (ensuring correct metadata)
    if os.path.abspath(path) == os.path.abspath(ref_raster):
        print(f"Copying reference raster: {name}")
        with rasterio.open(path) as src:
            meta = src.meta.copy()
            with rasterio.open(out_path, "w", **meta) as dst:
                dst.write(src.read(1), 1)
        continue

    print(f"Aligning {name}...")

    # Open source raster
    with rasterio.open(path) as src:
        src_data = src.read(1)
        src_transform = src.transform
        src_crs = src.crs
        src_dtype = src.dtypes[0]
        src_nodata = src.nodata

        # Prepare destination array (empty, filled with nodata)
        dst_array = np.full((ref_height, ref_width), src_nodata if src_nodata is not None else -9999, dtype=src_dtype)

        # Choose nearest for categorical rasters, bilinear for continuous
        if src_dtype.startswith("int"):
            resampling_method = Resampling.nearest
        else:
            resampling_method = Resampling.bilinear

        # Reproject & resample
        reproject(
            source=src_data,
            destination=dst_array,
            src_transform=src_transform,
            src_crs=src_crs,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            resampling=resampling_method,
            src_nodata=src_nodata,
            dst_nodata=src_nodata
        )

        # Write aligned raster
        out_meta = ref_meta.copy()
        out_meta.update({
            "dtype": src_dtype,
            "nodata": src_nodata,
            "count": 1
        })

        with rasterio.open(out_path, "w", **out_meta) as dst:
            dst.write(dst_array, 1)

print("Alignment complete. Rasters saved to:", out_dir)

Copying reference raster: elevation
Aligning df_rivers...
Aligning df_roads...
Aligning drainage_d...
Aligning lulc...
Aligning ndvi...
Aligning rainfall...
Aligning slope...
Aligning soil_type...
Aligning twi...
Alignment complete. Rasters saved to: D:\THESIS\Machine_Learning\rasters_aligned


In [29]:
raster_files = {
    "elevation": r"D:\THESIS\Machine_Learning\rasters_aligned\elevation_aligned.tif",
    "slope": r"D:\THESIS\Machine_Learning\rasters_aligned\slope_aligned.tif",
    "lulc": r"D:\THESIS\Machine_Learning\rasters_aligned\lulc_aligned.tif",
    "ndvi": r"D:\THESIS\Machine_Learning\rasters_aligned\ndvi_aligned.tif",
    "twi": r"D:\THESIS\Machine_Learning\rasters_aligned\twi_aligned.tif",
    "df_roads": r"D:\THESIS\Machine_Learning\rasters_aligned\df_roads_aligned.tif",
    "df_rivers": r"D:\THESIS\Machine_Learning\rasters_aligned\df_rivers_aligned.tif",
    "rainfall": r"D:\THESIS\Machine_Learning\rasters_aligned\rainfall_aligned.tif",
    "soil_type": r"D:\THESIS\Machine_Learning\rasters_aligned\soil_type_aligned.tif",
    "drainage_d": r"D:\THESIS\Machine_Learning\rasters_aligned\drainage_d_aligned.tif"
}

In [30]:
# quick_check.py
import os
import rasterio
from joblib import load

raster_dir = r"D:\THESIS\UNOSAT\rasters_aligned"
features = ["elevation","slope","lulc","ndvi","twi","df_roads","df_rivers","rainfall","soil_type","drainage_d"]
raster_files = {f: os.path.join(raster_dir, f"{f}_aligned.tif") for f in features}
missing = [p for p in raster_files.values() if not os.path.exists(p)]
if missing:
    print("Missing aligned rasters:", missing)
else:
    print("All aligned rasters found")

# print basic metadata for each
meta_list = []
for name, path in raster_files.items():
    with rasterio.open(path) as src:
        meta = {
            "name": name,
            "path": path,
            "width": src.width,
            "height": src.height,
            "crs": src.crs.to_string() if src.crs else None,
            "transform": tuple(src.transform),
            "nodata": src.nodata
        }
    meta_list.append(meta)
    print(name, "width,height:", meta["width"], meta["height"], "crs:", meta["crs"], "nodata:", meta["nodata"])

All aligned rasters found
elevation width,height: 2888 4098 crs: EPSG:4326 nodata: 32767.0
slope width,height: 2888 4098 crs: EPSG:4326 nodata: -3.4028230607370965e+38
lulc width,height: 2888 4098 crs: EPSG:4326 nodata: 255.0
ndvi width,height: 2888 4098 crs: EPSG:4326 nodata: -3.4028230607370965e+38
twi width,height: 2888 4098 crs: EPSG:4326 nodata: -3.4028230607370965e+38
df_roads width,height: 2888 4098 crs: EPSG:4326 nodata: -3.4028230607370965e+38
df_rivers width,height: 2888 4098 crs: EPSG:4326 nodata: -3.4028230607370965e+38
rainfall width,height: 2888 4098 crs: EPSG:4326 nodata: -3.4028230607370965e+38
soil_type width,height: 2888 4098 crs: EPSG:4326 nodata: 0.0
drainage_d width,height: 2888 4098 crs: EPSG:4326 nodata: -3.4028230607370965e+38


In [31]:
# quick consistency check
first = meta_list[0]
bad = [m["name"] for m in meta_list if (m["width"]!=first["width"] or m["height"]!=first["height"] or m["crs"]!=first["crs"] or m["transform"]!=first["transform"])]
if bad:
    print("Mismatch found in:", bad)
else:
    print("All rasters match exactly, good to proceed")

All rasters match exactly, good to proceed


In [32]:
# check model exists
model_path = r"D:\THESIS\Machine_Learning\ml_models\best_flood_model.joblib"  # update if different
if os.path.exists(model_path):
    try:
        load(model_path)
        print("Best model exists and can be loaded:", model_path)
    except Exception as e:
        print("Model file exists but failed to load, error:", e)
else:
    print("Best model not found at", model_path, "Run training and evaluation parts first.")

Best model exists and can be loaded: D:\THESIS\Machine_Learning\ml_models\best_flood_model.joblib


In [33]:
# part4_apply_model_aligned.py
import rasterio
from rasterio.windows import Window
import numpy as np
import pandas as pd
from joblib import load
import os
from tqdm import tqdm

In [34]:
# ---------- UPDATE PATHS ----------
model_path = r"D:\THESIS\Machine_Learning\ml_models\best_flood_model.joblib"
raster_dir = r"D:\THESIS\Machine_Learning\rasters_aligned"
out_path = r"D:\THESIS\Machine_Learning\output\flood_probability.tif"
os.makedirs(os.path.dirname(out_path), exist_ok=True)

In [35]:
features = ["elevation","slope","lulc","ndvi","twi","df_roads","df_rivers","rainfall","soil_type","drainage_d"]
raster_files = {f: os.path.join(raster_dir, f"{f}_aligned.tif") for f in features}

# load model
model = load(model_path)


In [36]:
# open sources and validate
srcs = [rasterio.open(raster_files[f]) for f in features]
w = srcs[0].width; h = srcs[0].height; tr = srcs[0].transform; crs = srcs[0].crs
for s in srcs[1:]:
    if s.width != w or s.height != h or s.transform != tr or s.crs != crs:
        raise ValueError("Raster geometry mismatch, aborting")

meta = srcs[0].meta.copy()
meta.update(count=1, dtype='float32', compress='lzw')

In [37]:
# windowing parameters, tune if needed
block_h = 512
block_w = 512

with rasterio.open(out_path, 'w', **meta) as dst:
    for i in tqdm(range(0, h, block_h), desc="rows"):
        win_h = min(block_h, h - i)
        for j in range(0, w, block_w):
            win_w = min(block_w, w - j)
            window = Window(j, i, win_w, win_h)

            arrs = []
            mask = None
            for s in srcs:
                a = s.read(1, window=window).astype(np.float32)
                arrs.append(a)
                nd = s.nodata
                layer_mask = np.isnan(a) | (a == -9999)
                if nd is not None:
                    layer_mask |= (a == nd)
                if mask is None:
                    mask = layer_mask
                else:
                    mask |= layer_mask

            valid_idx = np.where(~mask)
            if valid_idx[0].size == 0:
                out_block = np.full((win_h, win_w), srcs[0].nodata if srcs[0].nodata is not None else np.nan, dtype=np.float32)
                dst.write(out_block, 1, window=window)
                continue

            # assemble feature matrix
            cols = [arrs[k][valid_idx] for k in range(len(features))]
            X = np.vstack(cols).T
            # predict probabilities
            if hasattr(model, "predict_proba"):
                probs = model.predict_proba(pd.DataFrame(X, columns=features))[:, 1]
            else:
                probs = model.predict(pd.DataFrame(X, columns=features)).astype(float)

            out_block = np.full((win_h, win_w), np.nan, dtype=np.float32)
            out_block[valid_idx] = probs
            dst.write(out_block, 1, window=window)

rows: 100%|██████████████████████████████████████████████████████████████████████████████| 9/9 [00:23<00:00,  2.66s/it]


In [38]:
# close datasets
for s in srcs:
    s.close()

print("Probability map written to:", out_path)

Probability map written to: D:\THESIS\Machine_Learning\output\flood_probability.tif


In [39]:
# variable_importance.py
import pandas as pd
import numpy as np
from joblib import load
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

In [40]:
# ---------- UPDATE PATHS ----------
model_dir = r"D:\THESIS\Machine_Learning\ml_models"
test_csv  = r"D:\THESIS\Machine_Learning\ml_inputs\test_clean.csv"
out_dir   = r"D:\THESIS\Machine_Learning\importance_plots"

In [41]:
# features list
features = [
    "elevation","slope","lulc","ndvi","twi",
    "df_roads","df_rivers","rainfall","soil_type","drainage_d"
]

In [42]:
# Load testing data
df = pd.read_csv(test_csv)
X_test = df[features]
y_test = df["Class"].astype(int)

In [43]:
# Load models
rf = load(os.path.join(model_dir, "rf_pipe.joblib"))
gb = load(os.path.join(model_dir, "gb_pipe.joblib"))
svm = load(os.path.join(model_dir, "svm_pipe.joblib"))

models = {
    "RF": rf,
    "GBM": gb,     # GradientBoostingClassifier
    "SVM": svm
}

In [45]:
# Loop and compute permutation importance
for name, model in models.items():
    print(f"Computing importance for {name} ...")
    
    result = permutation_importance(
        model, 
        X_test, 
        y_test, 
        n_repeats=20, 
        random_state=42, 
        scoring="accuracy"
    )

    importances = result.importances_mean
    prd = (importances / importances.sum()) * 100     # PRD percent

    # Sort
    sorted_idx = np.argsort(prd)
    sorted_features = np.array(features)[sorted_idx]
    sorted_prd = prd[sorted_idx]

    # Plot
    plt.figure(figsize=(6, 5))
    plt.barh(sorted_features, sorted_prd, color="red")
    plt.xlabel("PRD (%)")
    plt.title(f"{name} Variable Importance")
    plt.grid(axis="x", linestyle="--", alpha=0.4)
    plt.tight_layout()

    # Save
    out_file = os.path.join(out_dir, f"{name}_importance.png")
    plt.savefig(out_file, dpi=300)
    plt.close()

    print("Saved:", out_file)

Computing importance for RF ...
Saved: D:\THESIS\Machine_Learning\importance_plots\RF_importance.png
Computing importance for GBM ...
Saved: D:\THESIS\Machine_Learning\importance_plots\GBM_importance.png
Computing importance for SVM ...
Saved: D:\THESIS\Machine_Learning\importance_plots\SVM_importance.png


In [46]:
# plot_rocs_threepanel.py
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from joblib import load
from sklearn.metrics import roc_curve, auc

In [47]:
# ----------------- UPDATE PATHS -----------------
model_dir = r"D:\THESIS\Machine_Learning\ml_models"
train_csv = r"D:\THESIS\Machine_Learning\ml_inputs\train_clean.csv"
test_csv  = r"D:\THESIS\Machine_Learning\ml_inputs\test_clean.csv"
out_fig   = r"D:\THESIS\Machine_Learning\outputs\roc_threepanel.png"
# model filenames (as saved earlier)
models_map = {
    "GBM": os.path.join(model_dir, "gb_pipe.joblib"),   # GB = BRT in your figures
    "RF":  os.path.join(model_dir, "rf_pipe.joblib"),
    "SVM": os.path.join(model_dir, "svm_pipe.joblib")
}

In [48]:
# features list used for training
features = [
    "elevation","slope","lulc","ndvi","twi",
    "df_roads","df_rivers","rainfall","soil_type","drainage_d"
]

In [49]:
def load_data():
    train = pd.read_csv(train_csv)
    test  = pd.read_csv(test_csv)
    X_train = train[features]
    y_train = train["Class"].astype(int)
    X_test  = test[features]
    y_test  = test["Class"].astype(int)
    return X_train, y_train, X_test, y_test

In [50]:
def get_scores(model, X):
    """
    Return a continuous score for ROC, prefer predict_proba[:,1], 
    fallback to decision_function if predict_proba unavailable.
    """
    try:
        scores = model.predict_proba(X)[:, 1]
    except Exception:
        # decision_function may return shape (n,) or (n,2). Reduce to one-dim if needed
        df = model.decision_function(X)
        if df.ndim == 2:
            # take positive class column if present
            scores = df[:, 1]
        else:
            scores = df
    return scores


In [51]:
# load data
X_train, y_train, X_test, y_test = load_data()

# loop models and save one ROC png per model
for name, model_path in models_map.items():
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file not found: {model_path}")
    model = load(model_path)
    # scores
    s_train = get_scores(model, X_train)
    s_test = get_scores(model, X_test)
    # compute ROC and AUC
    fpr_tr, tpr_tr, _ = roc_curve(y_train, s_train)
    fpr_te, tpr_te, _ = roc_curve(y_test, s_test)
    auc_tr = auc(fpr_tr, tpr_tr)
    auc_te = auc(fpr_te, tpr_te)

In [52]:
# compute ROC data for each model for train and test
roc_data = {}
for name, model in models.items():
    s_train = get_scores(model, X_train)
    s_test  = get_scores(model, X_test)
    fpr_tr, tpr_tr, _ = roc_curve(y_train, s_train)
    fpr_te, tpr_te, _ = roc_curve(y_test, s_test)
    auc_tr = auc(fpr_tr, tpr_tr)
    auc_te = auc(fpr_te, tpr_te)
    roc_data[name] = {
        "fpr_tr": fpr_tr, "tpr_tr": tpr_tr, "auc_tr": auc_tr,
        "fpr_te": fpr_te, "tpr_te": tpr_te, "auc_te": auc_te
    }


In [53]:
# loop models and save one ROC png per model
for name, model_path in models_map.items():
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file not found: {model_path}")
    model = load(model_path)
    # scores
    s_train = get_scores(model, X_train)
    s_test = get_scores(model, X_test)
    # compute ROC and AUC
    fpr_tr, tpr_tr, _ = roc_curve(y_train, s_train)
    fpr_te, tpr_te, _ = roc_curve(y_test, s_test)
    auc_tr = auc(fpr_tr, tpr_tr)
    auc_te = auc(fpr_te, tpr_te)

    # plot
    plt.figure(figsize=(6, 5))
    plt.plot(fpr_tr, tpr_tr, color="red", linewidth=2, label=f"AUC (training) = {auc_tr:.3f}")
    plt.plot(fpr_te, tpr_te, color="blue", linewidth=2, label=f"AUC (test.dep) = {auc_te:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)
    plt.xlim(0, 1)
    plt.ylim(0, 1.03)
    plt.xlabel("1-Specificity (false positive rate)")
    plt.ylabel("Sensitivity (true positive rate)")
    plt.title(f"ROC plot ({name})")
    leg = plt.legend(loc="lower right", frameon=True, fontsize=9)
    plt.tight_layout()

    out_file = os.path.join(out_dir, f"{name}_ROC.png")
    plt.savefig(out_file, dpi=300, bbox_inches="tight")
    plt.close()
    print("Saved:", out_file)

print("All ROC plots saved to:", out_dir)

Saved: D:\THESIS\Machine_Learning\importance_plots\GBM_ROC.png
Saved: D:\THESIS\Machine_Learning\importance_plots\RF_ROC.png
Saved: D:\THESIS\Machine_Learning\importance_plots\SVM_ROC.png
All ROC plots saved to: D:\THESIS\Machine_Learning\importance_plots
